# Landsat Collection 2 Level-2 Science Products


* **Products used:** 
[ls45_c2l2_sp](http://localhost:81/products/ls45_c2l2_sp),
[ls7_c2l2_sp](http://localhost:81/products/ls7_c2l2_sp),
[ls89_c2l2_sp](http://localhost:81/products/ls89_c2l2_sp)

Notebook modified from https://github.com/digitalearthafrica/deafrica-sandbox-notebooks

## Background

The United States Geological Survey's (USGS) [Landsat satellite program](https://www.nasa.gov/mission_pages/landsat/overview/index.html) has been capturing images of the globe for more than 30 years. These data are highly useful for land and coastal mapping studies. Landsat Collection 2 Level-2 Science Products provide [surface reflectance](https://www.usgs.gov/landsat-missions/landsat-collection-2-surface-reflectance) and [surface temperature](https://www.usgs.gov/landsat-missions/landsat-collection-2-surface-temperature). For a detailed description of the products, use the urls.

The present Landsat product forms a single, cohesive Analysis Ready Data (ARD) package, which allows you to analyse surface reflectance data as-is without the need to apply additional corrections.

![](./figures/landsat_timeline.png)

**Important details:**  
In order to convert given Digital Number (DN) values into proper Surface Reflectance (SR), you will need to restrict to  `Valid Range` and apply `Multiplicative Scale Factor` and `Additive Offset` which values vary depending on bands. Those values can be found in online documentation:  
    
   * [Landsat 8-9 Collection 2 (C2) Level 2 Science Product (L2SP) Guide](https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/s3fs-public/media/files/LSDS-1619_Landsat8-9-Collection2-Level2-Science-Product-Guide-v6.pdf#page=18): <a href="./figures/ls89_c2_l2_sp_scaling_factors.png" target="_blank" title="Open table">Table 6-1 🔎</a> (click on the magnifying glass to view the table)
   * [Landsat 4-7 Collection 2 Level 2 Science Product Guide](https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/s3fs-public/media/files/LSDS-1618_Landsat-4-7_C2-L2-ScienceProductGuide-v4.pdf#page=18): <a href="./figures/ls457_c2_l2_sp_scaling_factors.png" target="_blank" title="Open table">Table 1-1 🔎</a>

   For example the `blue band` of `ls89_c2l2_sp` (`SR_B2`) values need to be restrited to the `Valid Range: 7,273 - 43,636` and then use the formula `sr = dn * 2.75e-5 - 0.2`.
   ![](./figures/ls89_c2_l2_sp_scaling_factors_sr_b2.png)
   
   You will implement this concept in coming cells of this notebook.

**Date-range**:  
Depends on cube creation parameters, but collection covers 1984 to Present.

## Description

This notebook will run through loading in Landsat Collection 2 Level-2 Science Products images.
Topics covered include:

* Using the native `dc.load()` function to load in Landsat data
* Convert from Digital Number (DN) to Surface Reflectance (SR) and Surface Temperature (Kelvin and Celsius)

***

## Getting started

To run this analysis, run all the cells in the notebook, starting with the "Load packages" cell.

### Load packages

In [ ]:
import sys
sys.path.insert(1, '../utils/')

In [ ]:
# reload module before executing code
%load_ext autoreload
%autoreload 2
%matplotlib inline

import psutil
import time
import datacube
import numpy as np
import matplotlib.pyplot as plt
from dask.distributed import Client
from odc.geo import BoundingBox
from IPython.display import HTML, display
from utils.deafrica_plotting import display_map, rgb
from utils.nostra_mapping import create_map, last_drawn_rectangle, get_utm_epsg_code
from utils.nostra_masking import qa_pixel_mask

from planetary_computer import sign_url  # needed to access planetary-computer collection

### Connect to the datacube

In [ ]:
dc = datacube.Datacube(app="Landsat_Science_Products")

## Available products and measurements

### List products

In [ ]:
# We can use datacube's list_measurements functionality to list cubes products that are available.

dc_measurements = dc.list_measurements()
products = set([item[0] for item in dc_measurements.index])
for product in products:
    print(product)

In [ ]:
# Let's inspect one of the three Landsat products available (starting with "ls").

product = 'ls89_c2l2_sp'
dc.list_measurements().loc[product]

### Dask dashboard  
In order to optimise usage of your hardware it is recommended to run the next cells to configure Daskerisation and access Dask dashboard, which will allow you to monitor your hardware usage as well as Dask computation progress.  
![](./figures/dask_dashboard.png)

In [ ]:
# Get the available cpu(s) number and memory in gigabytes (to properly configure Dask client in follwoing cell)

print(f"Available cpu(s): {psutil.cpu_count()}")

available_memory = psutil.virtual_memory().available
available_memory_gb = available_memory / (1024 ** 3)
print(f"Available memory: {available_memory_gb:.2f} GB")

In [ ]:
try:
    client.dashboard_link
except:
    client = Client(n_workers=4, memory_limit='4GB')  # memory_limit PER WORKER !
client

## Load Landsat Surface Reflectance

Now that we know what products and measurements are available for the products, we can load data from the datacube using `dc.load`.

In the example below, we will load Surface Reflectance from `ls89_c2l2_sp` product. By specifying `output_crs='EPSG:6933'` and `resolution=30`, we request that datacube reproject our data to the African Albers coordinate reference system (CRS), with 30 x 30 m pixels. Finally, `group_by='solar_day'` ensures that overlapping images taken within seconds of each other as the satellite passes over are combined into a single time step in the data.

In [ ]:
# Check if default bbox is contained within the datacube
# and allow user to draw bbox if not.

# configure a default bounding box and visualize it
lat, lon = 22.821, 28.518
buffer = 0.05
default_bbox = BoundingBox(left=lon - buffer, bottom=lat - buffer,
                           right=lon + buffer, top=lat + buffer)

# get product bounding box
def bbox_union(bboxes):
    """
    Compute the union of a collection of BoundingBox objects.
    """
    bboxes = list(bboxes)
    if not bboxes:
        raise ValueError("No bounding boxes provided")

    left = min(b.left for b in bboxes)
    bottom = min(b.bottom for b in bboxes)
    right = max(b.right for b in bboxes)
    top = max(b.top for b in bboxes)

    return BoundingBox(left, bottom, right, top)
def dss_to_bbox(dss):
    geoms = [ds.extent.to_crs('epsg:4326') for ds in dss]
    return bbox_union(g.boundingbox for g in geoms)
dss = dc.find_datasets(product=product)
product_bbox = dss_to_bbox(dss)

is_contained =(default_bbox.left >= product_bbox.left and
               default_bbox.right <= product_bbox.right and
               default_bbox.bottom >= product_bbox.bottom and
               default_bbox.top <= product_bbox.top
              )

if is_contained:
    aoi_poly = (lon - buffer, lat - buffer, lon + buffer, lat + buffer)
    m = display_map(x=(aoi_poly[0], aoi_poly[2]), y=(aoi_poly[1], aoi_poly[3]))
    display(m)
else:
    m = create_map(tuple(product_bbox))
    display(m)

In [ ]:
# If needed remains user to draw a AoI
if not is_contained:
    aoi_poly = last_drawn_rectangle
    if len(aoi_poly) == 0:
        # Change the background color of the output area to light red
        display(HTML('''
            <div style="background-color: red; padding: 10px; border-radius: 5px;">
                The area of interest polygon (aoi_poly) has not been created. Please draw it in the previous cell.
            </div>
        '''))
    else:
        aoi_poly = aoi_poly.get('bounds')

In [ ]:
# get EPSG code for the center of the AoI

epsg_code = get_utm_epsg_code((aoi_poly[1] + aoi_poly[3]) / 2, (aoi_poly[0] + aoi_poly[2]) / 2)

In [ ]:
times = [ds.time.begin for ds in dss]

start_date = min(times)
end_date = max(times)

In [ ]:
import ipywidgets as widgets
start_date = widgets.DatePicker(description='Start date',
                                value = min(times).date(),
                                disabled=False)
end_date = widgets.DatePicker(description='End date',
                                value = max(times).date(),
                                disabled=False)
display(widgets.Label('IF REQUIRED define time period (cannot be outside of the initial displayed time) and run the next cell:'),
        widgets.HBox([start_date, end_date]))

In [ ]:
# load data
lazy_ds = dc.load(product="ls89_c2l2_sp",
                  measurements=['red', 'green', 'blue', 'qa_pixel'],
                  output_crs=f"EPSG:{epsg_code}",
                  y=(aoi_poly[1], aoi_poly[3]),
                  x=(aoi_poly[0], aoi_poly[2]),
                  time=(start_date.value.strftime('%Y-%m-%d'), end_date.value.strftime('%Y-%m-%d')),
                  resolution=30,
                  group_by="solar_day",
                  dask_chunks={},
                  patch_url=sign_url,
                  )
# customized chunking is less buggy when performed after loading !!!
lazy_ds = lazy_ds.chunk({"x": 512, "y": 512, "time": 1})
print(lazy_ds)

***
So far only metadata was accessed (which explains why loading was fast) as we used `dask_chunks` option, which is called "lazy loading". The `rgb` function in next cell will load the data which will slow down the process (you will find more Dask example in [Sentinel_2.ipynb](Sentinel_2.ipynb)). If you opened the Dask client link above you will be able to monitor activity, and possibly adapt chunking parameters (in previous cell) for better efficiency and stability.
***

In [ ]:
# Plot a composite of all scenes

rgb(lazy_ds, col="time", col_wrap=4)

In [ ]:
%%time

# Plotting composite took time as the dataset needed to be computed by the function (in other words data
# (no metadata) needed to be accessed).
# As histograms in coming cell, imply to access the data, let's compute lazy_ds for once, which will avoid
# to re-compute it later and save time.

ds = lazy_ds.compute()

## Remove pixels considered as nodata

In [ ]:
mask_arr = (qa_pixel_mask(ds['qa_pixel'],'cloud') | 
            qa_pixel_mask(ds['qa_pixel'],'cloud_shadow') |
            qa_pixel_mask(ds['qa_pixel'],'dilated_cloud') |
            qa_pixel_mask(ds['qa_pixel'],'cirrus') |
            qa_pixel_mask(ds['qa_pixel'],'nodata'))
ds = ds.where(~mask_arr).drop('qa_pixel')
ds = ds.dropna('time', how='all')  # Remove empty scenes

In [ ]:
# Plot a composite of all scenes

rgb(ds, col="time", col_wrap=4)

## Convert Digital Numbers (DN) to Surface Reflectance (SR)

In [ ]:
# Visualize distribution of each band

for v in ds.var():
    plt.hist(ds[v].values.flatten(), bins=50, alpha=0.5, label=v, color=v)
plt.legend()
plt.show()

In [ ]:
# First we need to make sure values are restricted to Valid Range. As selected bands have all the same
# Valid Range values we can do it at once

ds = ds.where((ds >= 7273) & (ds <= 43636), drop=True)

for v in ds.var():
    plt.hist(ds[v].values.flatten(), bins=50, alpha=0.5, label=v, color=v)
plt.legend()
plt.show()

In [ ]:
# Then convert DN to SR (with will give values between 0 and 1)

ds_sr = ds * 2.75e-5 - 0.2

for v in ds_sr.var():
    plt.hist(ds_sr[v].values.flatten(), bins=50, alpha=0.5, label=v, color=v)
plt.legend()
plt.show()

In [ ]:
# Re-plotting a composite of all scenes should give approximately the same result
# (except for the values lost when restricting to Valid Range)

rgb(ds_sr, col="time", col_wrap=4)

## Load Landsat Surface Temperature

In the example below, we will load Surface Temperature, using almost the same parameters as previously. In such case it is recommended to create a query object with common parameters to be re-used. Let's do it for the demo.

In [ ]:
# Create a query object

common_query = {
    'product': 'ls89_c2l2_sp',
    'output_crs': f"EPSG:{epsg_code}",
    'y': (aoi_poly[1], aoi_poly[3]),
    'x': (aoi_poly[0], aoi_poly[2]),
    'time': (start_date.value.strftime('%Y-%m-%d'), end_date.value.strftime('%Y-%m-%d')),
    'resolution': 30,
    'group_by': "solar_day",
    'patch_url': sign_url,
}

In [ ]:
# Then load data using common_query
lazy_ds = dc.load(measurements=['surface_temperature', 'qa_pixel'],  # alias can also be used
                  dask_chunks={},
                  **common_query)

# customized chunking is less buggy when performed after loading !!!
lazy_ds = lazy_ds.chunk({"x": 512, "y": 512, "time": 1})

print(lazy_ds)

In [ ]:
# compute Lazy_ds as plotting the dataset will require access to data

ds = lazy_ds.compute()

In [ ]:
# plot surface temperature

ds['time'] = ds['time'].astype(str)  # Fix KeyError with time format with imshow
ds.surface_temperature.plot.imshow(col='time', col_wrap=4, cmap='coolwarm')

In [ ]:
# Remove nodata pixels using the mask criteria previously used

ds = ds.where(~mask_arr).drop('qa_pixel')
ds = ds.dropna('time', how='all')  # Remove empty scenes

***
Even if the legende above mention kelvin, `lazy_ds` values need to be constrained and scaled (as Surface Temperature above but with different values). Let's do it in the next cell and plot again.
***

In [ ]:
# restrict to Valid Range
ds = ds.where((ds >= 293) & (ds <= 61440), drop=True)

# convert DN to ST (which will give Kelvins)
st = ds * 3.41802e-3 + 149

st['time'] = st['time'].astype(str)  # Fix KeyError with time format with imshow
st.surface_temperature.plot.imshow(col='time', col_wrap=4, cmap='coolwarm')

In [ ]:
# now convert kelvin into celsius

st = st - 273.15
st.surface_temperature.plot.imshow(col='time', col_wrap=4, cmap='coolwarm')

In [ ]:
# In case st contains negative and positive values imshow will center
# the colormap on zero, to avoid that you need to manually set vmin and vmax arguments

st.surface_temperature.plot.imshow(col='time', col_wrap=4, cmap='coolwarm',
                                   vmin = st.surface_temperature.min().values,
                                   vmax = st.surface_temperature.max().values)

***

## Additional information

**License:** The code in this notebook is slighly modified from https://github.com/digitalearthafrica/deafrica-sandbox-notebooks and licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0).

**Compatible datacube version:**

In [ ]:
print(datacube.__version__)

**Last tested:**

In [ ]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')

In [ ]:
!pip freeze